# 04. ContextBuilder

В курсе мы будем часто возвращаться к простой идее: хороший агент начинается не с магического класса `Agent`, а с явного сборщика контекста.

До кэпстоуна используем нейтральный docs-like сценарий: support-агент готовит ответ по возврату заказа и должен выбрать только нужные части контекста.

Здесь мы напишем маленький `ContextBuilder`, который принимает секции, сортирует их по приоритету и возвращает готовый пакет для следующего шага.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json


@dataclass
class ContextSection:
    name: str
    content: str
    tokens: int
    priority: int
    trust: str


class ContextBuilder:
    def __init__(self, token_budget: int):
        self.token_budget = token_budget
        self.sections = []

    def add(self, section: ContextSection):
        self.sections.append(section)

    def build(self):
        included = []
        dropped = []
        used = 0

        for section in sorted(self.sections, key=lambda s: s.priority, reverse=True):
            if used + section.tokens <= self.token_budget:
                included.append(section)
                used += section.tokens
            else:
                dropped.append(section)

        return {
            "token_budget": self.token_budget,
            "tokens_used": used,
            "included": [section.__dict__ for section in included],
            "dropped": [section.__dict__ for section in dropped],
        }


builder = ContextBuilder(token_budget=360)
builder.add(ContextSection("rules", "Не писать без preview и источников.", 80, 100, "high"))
builder.add(ContextSection("task", "Подготовить ответ по возврату ORD-1842.", 35, 95, "medium"))
builder.add(ContextSection("case_state", "step=draft_reply; approval_required=true", 60, 90, "high"))
builder.add(ContextSection("refund_policy", "Возврат возможен в течение 30 дней, если товар не использовался.", 180, 80, "workspace"))
builder.add(ContextSection("support_log", "Очень длинный лог переписки клиента и поддержки.", 260, 45, "tool_bounded"))

packet = builder.build()
print(json.dumps(packet, ensure_ascii=False, indent=2))

Теперь сохраним пакет как артефакт запуска. В настоящем кэпстоун-проекте такие артефакты помогают видеть, почему агент принял решение и какой контекст был у модели.

In [ ]:
out_dir = Path("artifacts/context")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "demo_run_context.json"
out_path.write_text(json.dumps(packet, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Сохранено: {out_path}")